# Arm G boundary geometry — is writability distance, or responsiveness?

Three papers disagree about whether a probe direction is a control point. Roy et al.
report **0% correction** for hallucination in 7/7 models; a multi-behaviour study on
**this same model at layer 15** reports hallucination *highly steerable* (+60); Rift
reports deception neither inducible nor correctable. Our own dose run moved the
deciding rows from a minimum margin of +2.125 to +0.125 and flipped nothing.

Hypothesis: these measure **distance to a decision boundary**, not causal role.

**The naive test is circular.** With a shared slope s, dose-to-flip is exactly
`−margin(0)/s`, so regressing dose-to-flip on baseline margin returns R²=1 as
*arithmetic*. The self-test confirms this. The content is the decomposition

```
dose_to_flip_i  =  −baseline_margin_i / responsiveness_i
```

and specifically whether **responsiveness** is homogeneous. Homogeneous → writability
is boundary distance and the literature's disagreement is a task-design artifact.
Systematically varying → some states are intrinsically harder to move.

**Primary test needs no row to flip.** Per-row slopes are estimable from all 128 rows
regardless of crossings, so low crossing coverage qualifies the secondary R² fit but
cannot make the run uninformative.

Signed additive steering `x ← x + c·σ·r` at layer 16, all positions, c ∈ ±{0.5,1,2,3,4,6,8}.
Signed doses give induce and correct on one axis. Matched-dose random directions and a
coherence gate throughout.

Set **Runtime → Change runtime type → A100 GPU**. ~15 doses plus controls; a few minutes.

In [ ]:
# Colab supplies torch/CUDA.
print("Protocol: ARM_G_BOUNDARY_V1")
%pip -q install "transformers==5.0.0" "accelerate==1.12.0" \
  "sentence-transformers==5.2.2" "scikit-learn==1.8.0"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
LAUNCH_DIR = "/content/drive/MyDrive/phi-map/arm-g-boundary-launch"
LAUNCH_FILES = (
    "arm_g_boundary.py",
    "arm_g_cross_layer.py",
    "arm_g_causal_subspace.py",
    "arm_g_causal_dose_ablation.py",
    "arm_g_causal.py",
    "arm_g_phase1.py",
    "arm_g_scenarios.py",
)
for name in LAUNCH_FILES:
    source = f"{LAUNCH_DIR}/{name}"
    assert os.path.exists(source), f"Missing {source}"
    shutil.copy2(source, f"/content/{name}")
print("Arm G boundary launch files staged: OK")

In [ ]:
ACTING_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
WORK_DIR = "/content/drive/MyDrive/phi-map/arm-g-boundary-seed110-v1"
PAIRS_PER_FAMILY = 16
BOOTSTRAP = 2000
RANDOM_DIRECTIONS = 4
BATCH_SIZE = 16

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
assert HF_TOKEN, "Add an HF_TOKEN secret with Llama-3.1-8B-Instruct access"

In [ ]:
from huggingface_hub import hf_hub_download
hf_hub_download(ACTING_MODEL, "config.json", token=HF_TOKEN)
print("Hugging Face model access: OK")

import subprocess, torch
subprocess.run(["nvidia-smi"], check=True)
props = torch.cuda.get_device_properties(0)
print(props.name, round(props.total_memory / 1024**3, 1), "GiB")
assert "A100" in props.name and props.total_memory >= 35 * 1024**3
assert torch.cuda.is_bf16_supported()

In [ ]:
# Deterministic tests: additive-hook geometry, exact crossing-dose recovery,
# the shared-slope R^2=1 circularity, and all four decision branches.
import os, subprocess, sys
env = dict(os.environ)
env["HF_TOKEN"] = HF_TOKEN
base_cmd = [
    sys.executable, "/content/arm_g_boundary.py",
    "--model", ACTING_MODEL,
    "--output-dir", WORK_DIR,
    "--pairs-per-family", str(PAIRS_PER_FAMILY),
    "--bootstrap", str(BOOTSTRAP),
    "--random-directions", str(RANDOM_DIRECTIONS),
    "--batch-size", str(BATCH_SIZE),
]
subprocess.run(base_cmd + ["--self-test"], check=True, env=env)

In [ ]:
# Signed dose sweep with per-row slope fits and matched-dose random controls.
subprocess.run(base_cmd, check=True, env=env)

In [ ]:
import json
result_path = f"{WORK_DIR}/arm_g_boundary_result.json"
result = json.load(open(result_path))
summary = {
    "decision": result["decision"],
    "decision_reasons": result["decision_reasons"],
    "decision_detail": result["decision_detail"],
    "projection_sigma": result["projection_sigma"],
    "baseline": result["baseline"],
    "dose_response": result["dose_response"],
    "decomposition": result["decomposition"],
    "sample_counts": result["sample_counts"],
}
print(json.dumps(summary, indent=2))

In [ ]:
import base64, gzip, json

summary_path = f"{WORK_DIR}/arm_g_boundary_result_summary.json"
archive_path = f"{WORK_DIR}/arm_g_boundary_result.json.gz.b64"
with open(summary_path, "w") as h:
    json.dump({**summary, "random_controls": result["random_controls"],
        "protocol": {"source_seeds": [101, 102], "evaluation_seed": 110,
        "direction_layer": 16, "bootstrap_repetitions": BOOTSTRAP},
        "full_result_artifact": "arm_g_boundary_result.json.gz.b64"}, h, indent=1)
raw = json.dumps(result).encode("utf-8")
with open(archive_path, "wb") as h:
    h.write(base64.b64encode(gzip.compress(raw)))
print("summary:", summary_path)
print("archive:", archive_path)